# Metadata

Domain: Material Science

Subdomain: Crystallography

References: https://doi.org/10.1107/S056773947400043X (Grimmer, Bollmann and Warrington, Acta Cryst. A30, 197-207, 1974); https://doi.org/10.1016/0001-6160(66)90168-4 (Brandon, Acta Metallurgica 14, 1479-1484, 1966)


# Title

Coincidence-Site-Lattice Classification of Cubic Grain Boundaries from EBSD Orientations Using Brandon's Criterion


# Subproblem 1

## Prompt

When two identical cubic lattices are rotated relative to each other about a common lattice point, they share a periodic sublattice of coincident points (a coincidence-site lattice, CSL) only if the rotation matrix has rational entries in the cubic crystal axes. This is the starting point of CSL descriptions of grain boundaries in cubic metals and ceramics.

Consider a proper rotation matrix \\(R\\) (a real \\(3\\times 3\\) matrix with \\(R^{\\mathsf T}R=I\\) and \\(\\det R=+1\\)) expressed in the cubic crystal axes. \\(R\\) is called a \\(\\Sigma\\) rotation if \\(\\Sigma\\) is the smallest positive integer for which all nine entries of \\(\\Sigma R\\) are integers. Let \\(\\mathcal{S}_\\Sigma\\) denote the set of all \\(\\Sigma\\) rotations.

Implement a function that, for a given positive integer `sigma`, returns every \\(\\Sigma\\) rotation as the integer matrix \\(\\Sigma R\\). The result must be an integer NumPy array of shape `(K, 3, 3)` in which every rotation of \\(\\mathcal{S}_\\Sigma\\) appears exactly once. The matrices must be sorted in ascending lexicographic order of their row-major flattened 9-entry integer vectors (compare the first entries; if equal, the second entries; and so on). If \\(\\mathcal{S}_\\Sigma\\) is empty for the given `sigma`, return an empty integer array of shape `(0, 3, 3)`. Use only NumPy and the Python standard library.

```python
import numpy as np

def csl_rotation_set(sigma):
    """
    Enumerate all Sigma rotations of a cubic crystal.

    A Sigma rotation is a proper rotation matrix R (R^T R = I, det R = +1),
    expressed in the cubic crystal axes, for which Sigma is the smallest positive
    integer such that all nine entries of Sigma * R are integers.

    Input:
        sigma : int
            Positive integer (the CSL index Sigma).

    Output:
        rotations : numpy.ndarray of integer dtype, shape (K, 3, 3)
            rotations[k] = Sigma * R_k, where R_k is the k-th Sigma rotation.
            Every distinct Sigma rotation appears exactly once, and the matrices are
            sorted in ascending lexicographic order of rotations[k].reshape(9)
            (row-major flattening). If no Sigma rotation exists for the given sigma,
            K = 0 and the returned array has shape (0, 3, 3).
    """
    ...
```


## Background

**Rational rotations and the CSL index.** Two cubic lattices related by a rotation \\(R\\) about a common lattice point have a coincidence-site lattice if and only if \\(R\\) has rational entries in the cubic axes. Following the usual convention for cubic crystals, the CSL index \\(\\Sigma\\) is the smallest positive integer such that \\(\\Sigma R\\) is an integer matrix.

**Parametrization by integer quaternions.** Every rational rotation matrix can be generated by the Euler-Rodrigues formula from an integer quaternion \\(q=(a,b,c,d)\\) with \\(\\gcd(a,b,c,d)=1\\) (a primitive quaternion, unique up to an overall sign). With \\(N=a^2+b^2+c^2+d^2\\),

\\[
R(q)=\\frac{1}{N}\\begin{pmatrix}
a^2+b^2-c^2-d^2 & 2(bc-ad) & 2(bd+ac)\\\\
2(bc+ad) & a^2-b^2+c^2-d^2 & 2(cd-ab)\\\\
2(bd-ac) & 2(cd+ab) & a^2-b^2-c^2+d^2
\\end{pmatrix}.
\\]

The index of \\(R(q)\\) is the largest odd divisor of \\(N\\). Note that \\(N\\) itself need not equal \\(\\Sigma\\): the entries of \\(N\\,R(q)\\) may share a common power-of-two factor that has to be cancelled.

**Properties of \\(\\mathcal{S}_\\Sigma\\).**

- Each row of \\(\\Sigma R\\) is an integer vector of squared length \\(\\Sigma^2\\). If \\(\\Sigma\\) were even, \\(\\Sigma^2\\equiv 0 \\pmod 4\\) would force all entries of \\(\\Sigma R\\) to be even, contradicting the minimality of \\(\\Sigma\\). Hence \\(\\mathcal{S}_\\Sigma\\) is empty for every even \\(\\Sigma\\).
- \\(\\mathcal{S}_1\\) consists of the 24 proper rotations of the cube (signed permutation matrices with determinant \\(+1\\)).
- \\(\\mathcal{S}_\\Sigma\\) is closed under multiplication from the left or the right by any of these 24 rotations, and under transposition.
- The number of elements is \\(|\\mathcal{S}_\\Sigma|=24\\,\\psi(\\Sigma)\\) with \\(\\psi(\\Sigma)=\\Sigma\\prod_{p\\mid\\Sigma}(1+1/p)\\), where the product runs over the distinct prime divisors of \\(\\Sigma\\). For example, \\(|\\mathcal{S}_3|=96\\); it contains the \\(60^\\circ\\) rotation about \\([111]\\) (the \\(\\Sigma 3\\) twin), \\(R=\\tfrac{1}{3}\\begin{pmatrix}2&-1&2\\\\2&2&-1\\\\-1&2&2\\end{pmatrix}\\).

References: H. Grimmer, W. Bollmann and D. H. Warrington, Acta Cryst. A30, 197-207 (1974), https://doi.org/10.1107/S056773947400043X


## Testing Template

In [ ]:
import numpy as np
from itertools import permutations, product


def test_case_1():
    # Sigma = 1: the 24 proper rotations of the cube, sorted lexicographically
    expected = []
    for perm in permutations(range(3)):
        for signs in product([1, -1], repeat=3):
            m = np.zeros((3, 3), dtype=int)
            for row, (col, sgn) in enumerate(zip(perm, signs)):
                m[row, col] = sgn
            if round(np.linalg.det(m)) == 1:
                expected.append(m)
    expected = np.array(sorted(expected, key=lambda m: tuple(m.reshape(9))))
    out = csl_rotation_set(1)
    assert np.issubdtype(out.dtype, np.integer)
    assert out.shape == (24, 3, 3)
    assert np.array_equal(out, expected)

    # Sigma = 3: 96 rotations, first and last entries of the sorted list, and the 60 deg / [111] twin
    out3 = csl_rotation_set(3)
    assert np.issubdtype(out3.dtype, np.integer)
    assert out3.shape == (96, 3, 3)
    assert np.array_equal(out3[0], np.array([[-2, -2, -1], [-2, 1, 2], [-1, 2, -2]]))
    assert np.array_equal(out3[47], np.array([[-1, 2, 2], [2, 2, -1], [-2, 1, -2]]))
    assert np.array_equal(out3[-1], np.array([[2, 2, 1], [2, -1, -2], [-1, 2, -2]]))
    twin = np.array([[2, -1, 2], [2, 2, -1], [-1, 2, 2]])
    assert any(np.array_equal(m, twin) for m in out3)


def test_case_2():
    # Counts and algebraic properties for prime, composite and prime-power Sigma
    def psi(n):
        result, m, p = n, n, 2
        while p * p <= m:
            if m % p == 0:
                result = result // p * (p + 1)
                while m % p == 0:
                    m //= p
            p += 1
        if m > 1:
            result = result // m * (m + 1)
        return result

    for sigma in [5, 7, 9, 15, 21, 25, 27, 35, 45, 49]:
        out = csl_rotation_set(sigma)
        assert np.issubdtype(out.dtype, np.integer)
        assert out.shape == (24 * psi(sigma), 3, 3)
        flat = out.reshape(len(out), 9)
        # orthogonality: (Sigma R)(Sigma R)^T = Sigma^2 I
        gram = np.einsum("kij,klj->kil", out, out)
        assert np.all(gram == sigma ** 2 * np.eye(3, dtype=int))
        # proper rotation: det(Sigma R) = Sigma^3
        det = np.einsum("ki,ki->k", out[:, 0], np.cross(out[:, 1], out[:, 2]))
        assert np.all(det == sigma ** 3)
        # Sigma is minimal: no common factor of Sigma can be cancelled from all entries
        assert np.all(np.gcd(np.gcd.reduce(flat, axis=1), sigma) == 1)
        # sorted in ascending lexicographic order, no duplicates
        keys = [tuple(row) for row in flat]
        assert keys == sorted(keys)
        assert len(set(keys)) == len(keys)


def test_case_3():
    # Even Sigma: no Sigma rotation exists
    for sigma in [2, 4, 6, 8, 10, 12]:
        out = csl_rotation_set(sigma)
        assert np.issubdtype(out.dtype, np.integer)
        assert out.shape == (0, 3, 3)

    # Sigma = 13: closure under cubic symmetry (left and right) and under transposition
    cube = csl_rotation_set(1)
    out = csl_rotation_set(13)
    assert out.shape == (24 * 14, 3, 3)
    members = {tuple(m.reshape(9)) for m in out}
    for op in cube[::5]:
        for m in out[::7]:
            assert tuple((op @ m).reshape(9)) in members
            assert tuple((m @ op).reshape(9)) in members
            assert tuple(m.T.reshape(9)) in members


## Solution

In [ ]:
import numpy as np


def csl_rotation_set(sigma):
    sigma = int(sigma)
    if sigma < 1 or sigma % 2 == 0:
        return np.zeros((0, 3, 3), dtype=np.int64)

    # integer row vectors of squared length sigma^2
    r = np.arange(-sigma, sigma + 1, dtype=np.int64)
    x, y, z = np.meshgrid(r, r, r, indexing="ij")
    mask = x * x + y * y + z * z == sigma * sigma
    rows = np.stack([x[mask], y[mask], z[mask]], axis=1)

    # ordered pairs of orthogonal rows; the third row of a proper rotation is (r1 x r2) / sigma
    i_idx, j_idx = np.nonzero(rows @ rows.T == 0)
    r1, r2 = rows[i_idx], rows[j_idx]
    cross = np.cross(r1, r2)
    ok = np.all(cross % sigma == 0, axis=1)
    r1, r2, cross = r1[ok], r2[ok], cross[ok]
    mats = np.stack([r1, r2, cross // sigma], axis=1)

    # sigma must be the smallest integer that clears all denominators
    common = np.gcd.reduce(mats.reshape(len(mats), 9), axis=1)
    mats = mats[np.gcd(common, sigma) == 1]

    # ascending lexicographic order of the row-major flattened matrices
    flat = mats.reshape(len(mats), 9)
    order = np.lexsort(flat.T[::-1])
    return mats[order]


# Subproblem 2

## Prompt

In a cubic polycrystal, the misorientation across a grain boundary is a proper rotation matrix \\(M\\) (a real \\(3\\times 3\\) matrix with \\(M^{\\mathsf T}M=I\\) and \\(\\det M=+1\\)) expressed in the cubic crystal axes. Measured misorientations are never exactly coincident, so one needs the angular distance from \\(M\\) to the closest exact coincidence rotation.

A proper rotation matrix \\(R\\), expressed in the cubic crystal axes, is called a \\(\\Sigma\\) rotation if \\(\\Sigma\\) is the smallest positive integer for which all nine entries of \\(\\Sigma R\\) are integers. Let \\(\\mathcal{S}_\\Sigma\\) denote the set of all \\(\\Sigma\\) rotations. The rotation angle \\(\\theta(Q)\\in[0,\\pi]\\) of a proper rotation matrix \\(Q\\) is defined by \\(\\cos\\theta(Q)=\\tfrac{1}{2}(\\operatorname{tr}Q-1)\\).

For a given misorientation matrix \\(M\\) and a given odd positive integer \\(\\Sigma\\), the deviation from exact \\(\\Sigma\\) coincidence is

\\[
\\Delta\\theta(M,\\Sigma)=\\min_{R\\in\\mathcal{S}_\\Sigma}\\,\\theta\\!\\left(M R^{\\mathsf T}\\right).
\\]

Implement a function that returns \\(\\Delta\\theta(M,\\Sigma)\\) in degrees. The returned value must have an absolute error below \\(10^{-9}\\) degrees for every valid input, including near-exact coincidences whose true deviation is much smaller than \\(10^{-6}\\) degrees (as occurs when a bicrystal is constructed numerically). Use only NumPy and the Python standard library.

```python
import numpy as np

def csl_deviation(M, sigma):
    """
    Angular deviation of a misorientation from the closest exact Sigma rotation.

    A Sigma rotation is a proper rotation matrix R, expressed in the cubic crystal
    axes, for which Sigma is the smallest positive integer such that all nine
    entries of Sigma * R are integers. S_Sigma is the set of all Sigma rotations.

    Inputs:
        M : array_like of float, shape (3, 3)
            Proper rotation matrix (M^T M = I, det M = +1) in the cubic crystal axes.
            Dimensionless.
        sigma : int
            Odd positive integer (the CSL index Sigma).

    Output:
        deviation : float
            min over R in S_Sigma of the rotation angle of (M R^T), in degrees.
            The value lies in [0, 180] and its absolute error must be below 1e-9 degrees.
    """
    ...
```


## Background

**Angular distance between rotations.** For proper rotations \\(A\\) and \\(B\\), the rotation angle \\(\\theta(A B^{\\mathsf T})\\) is the geodesic (smallest-rotation) distance between them on \\(SO(3)\\). It is symmetric in \\(A\\) and \\(B\\) and is unchanged under \\(A,B\\to AC,BC\\) or \\(A,B\\to CA,CB\\) for any rotation \\(C\\). The deviation \\(\\Delta\\theta(M,\\Sigma)\\) is therefore the distance from \\(M\\) to the finite set \\(\\mathcal{S}_\\Sigma\\).

**Rational rotations.** A rotation matrix has rational entries if and only if it can be generated by the Euler-Rodrigues formula from an integer quaternion \\(q=(a,b,c,d)\\) with \\(\\gcd(a,b,c,d)=1\\) (unique up to overall sign). With \\(N=a^2+b^2+c^2+d^2\\),

\\[
R(q)=\\frac{1}{N}\\begin{pmatrix}
a^2+b^2-c^2-d^2 & 2(bc-ad) & 2(bd+ac)\\\\
2(bc+ad) & a^2-b^2+c^2-d^2 & 2(cd-ab)\\\\
2(bd-ac) & 2(cd+ab) & a^2-b^2-c^2+d^2
\\end{pmatrix},
\\]

and its index \\(\\Sigma\\) is the largest odd divisor of \\(N\\). The entries of \\(N\\,R(q)\\) can share a common power-of-two factor, so \\(N\\) is not always equal to \\(\\Sigma\\).

**Properties of \\(\\mathcal{S}_\\Sigma\\).** \\(\\mathcal{S}_\\Sigma\\) is empty for even \\(\\Sigma\\) (all entries of an integer row of squared length \\(\\Sigma^2\\) would be even, contradicting minimality). \\(\\mathcal{S}_1\\) is the set of the 24 proper rotations of the cube. \\(\\mathcal{S}_\\Sigma\\) is closed under left and right multiplication by these 24 rotations and under transposition, and it has \\(24\\,\\psi(\\Sigma)\\) elements with \\(\\psi(\\Sigma)=\\Sigma\\prod_{p\\mid\\Sigma}(1+1/p)\\) (product over distinct primes \\(p\\)). Consequently, \\(\\Delta\\theta(M,\\Sigma)\\) does not change when \\(M\\) is replaced by any crystallographically equivalent description \\(O_1 M O_2\\) (\\(O_1,O_2\\) proper cubic rotations) or by \\(M^{\\mathsf T}\\). For \\(\\Sigma=1\\), \\(\\Delta\\theta\\) is the disorientation angle.

**Numerical note.** Near-exact coincidence occurs in practice: bicrystals built for atomistic simulation reproduce a \\(\\Sigma\\) rotation to double precision, and deviations well below \\(10^{-6}\\) degrees must then be resolved with a small absolute error.

References: D. G. Brandon, Acta Metallurgica 14, 1479-1484 (1966), https://doi.org/10.1016/0001-6160(66)90168-4; H. Grimmer, W. Bollmann and D. H. Warrington, Acta Cryst. A30, 197-207 (1974), https://doi.org/10.1107/S056773947400043X


## Testing Template

In [ ]:
import numpy as np


def _rot(axis, angle_deg):
    # active rotation matrix about `axis` by `angle_deg` (Rodrigues' formula)
    n = np.asarray(axis, dtype=float)
    n = n / np.linalg.norm(n)
    t = np.radians(angle_deg)
    k = np.array([[0.0, -n[2], n[1]], [n[2], 0.0, -n[0]], [-n[1], n[0], 0.0]])
    return np.cos(t) * np.eye(3) + np.sin(t) * k + (1.0 - np.cos(t)) * np.outer(n, n)


def test_case_1():
    twin = np.array([[2, -1, 2], [2, 2, -1], [-1, 2, 2]]) / 3.0      # Sigma 3: 60 deg about [111]
    s5 = np.array([[5, 0, 0], [0, 4, -3], [0, 3, 4]]) / 5.0          # Sigma 5: 36.87 deg about [100]
    quarter_turn = np.array([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]])  # cubic symmetry operation
    # exact coincidence with the own Sigma
    assert abs(csl_deviation(np.eye(3), 1)) < 1e-9
    assert abs(csl_deviation(quarter_turn, 1)) < 1e-9
    assert abs(csl_deviation(twin, 3)) < 1e-9
    assert abs(csl_deviation(s5, 5)) < 1e-9
    # deviation from a different Sigma
    assert abs(csl_deviation(twin, 1) - 60.0) < 1e-9
    assert abs(csl_deviation(s5, 1) - np.degrees(2.0 * np.arctan(1.0 / 3.0))) < 1e-9
    assert abs(csl_deviation(twin, 9) - 31.58633809652793) < 1e-9
    assert abs(csl_deviation(s5, 3) - 48.18968510422140) < 1e-9


def test_case_2():
    twin = np.array([[2, -1, 2], [2, 2, -1], [-1, 2, 2]]) / 3.0
    s5 = np.array([[5, 0, 0], [0, 4, -3], [0, 3, 4]]) / 5.0
    s7 = _rot([1, 1, 1], np.degrees(2.0 * np.arctan(np.sqrt(3.0) / 5.0)))    # Sigma 7
    s9 = _rot([1, 1, 0], np.degrees(2.0 * np.arctan(np.sqrt(2.0) / 4.0)))    # Sigma 9
    # M = E R with a small rotation E of angle eps: the closest Sigma rotation is R, so the deviation is eps
    cases = [
        (twin, 3, [1, 2, -1], 0.25),
        (s5, 5, [3, -1, 2], 0.4),
        (s7, 7, [0.3, 0.5, 0.8], 0.35),
        (s9, 9, [1, -1, 4], 0.2),
        (np.eye(3), 1, [1, 1, 1], 1e-6),
        (twin, 3, [2, 1, 2], 1e-6),
        (s5, 5, [0, 0, 1], 1e-5),
        (s7, 7, [1, -2, 5], 1e-7),
    ]
    for rotation, sigma, axis, eps in cases:
        m = _rot(axis, eps) @ rotation
        assert abs(csl_deviation(m, sigma) - eps) < 1e-9


def test_case_3():
    # 45 deg about [001] is not a coincidence rotation for any Sigma
    m45 = _rot([0, 0, 1], 45.0)
    expected45 = {1: 45.0, 3: 48.83031675669113, 5: 8.130102354155978,
                  7: 38.07528692669672, 9: 32.00846711268371, 15: 21.08528914397513}
    for sigma, value in expected45.items():
        assert abs(csl_deviation(m45, sigma) - value) < 1e-9

    # generic misorientation (values verified with 50-digit arithmetic)
    m = _rot([1, 2, 3], 71.3)
    reference = {3: 22.11209974299143, 9: 17.24517516167680,
                 15: 14.87744627072566, 49: 12.01548983619133}
    for sigma, value in reference.items():
        assert abs(csl_deviation(m, sigma) - value) < 1e-9

    # invariance under equivalent descriptions: cubic operations on either side and transposition
    o1 = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]])
    o2 = np.array([[0.0, -1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, -1.0]])
    for sigma in [9, 15]:
        base = reference[sigma]
        assert abs(csl_deviation(o1 @ m @ o2, sigma) - base) < 1e-9
        assert abs(csl_deviation(m.T, sigma) - base) < 1e-9


## Solution

In [ ]:
import numpy as np


def csl_rotation_set(sigma):
    sigma = int(sigma)
    if sigma < 1 or sigma % 2 == 0:
        return np.zeros((0, 3, 3), dtype=np.int64)

    # integer row vectors of squared length sigma^2
    r = np.arange(-sigma, sigma + 1, dtype=np.int64)
    x, y, z = np.meshgrid(r, r, r, indexing="ij")
    mask = x * x + y * y + z * z == sigma * sigma
    rows = np.stack([x[mask], y[mask], z[mask]], axis=1)

    # ordered pairs of orthogonal rows; the third row of a proper rotation is (r1 x r2) / sigma
    i_idx, j_idx = np.nonzero(rows @ rows.T == 0)
    r1, r2 = rows[i_idx], rows[j_idx]
    cross = np.cross(r1, r2)
    ok = np.all(cross % sigma == 0, axis=1)
    r1, r2, cross = r1[ok], r2[ok], cross[ok]
    mats = np.stack([r1, r2, cross // sigma], axis=1)

    # sigma must be the smallest integer that clears all denominators
    common = np.gcd.reduce(mats.reshape(len(mats), 9), axis=1)
    mats = mats[np.gcd(common, sigma) == 1]

    # ascending lexicographic order of the row-major flattened matrices
    flat = mats.reshape(len(mats), 9)
    order = np.lexsort(flat.T[::-1])
    return mats[order]


def csl_deviation(M, sigma):
    M = np.asarray(M, dtype=float)
    S = csl_rotation_set(sigma).astype(float) / sigma
    Q = np.einsum("ij,kmj->kim", M, S)  # M @ R^T for every Sigma rotation R
    cos_t = 0.5 * (Q[:, 0, 0] + Q[:, 1, 1] + Q[:, 2, 2] - 1.0)
    sin_t = 0.5 * np.sqrt(
        (Q[:, 2, 1] - Q[:, 1, 2]) ** 2
        + (Q[:, 0, 2] - Q[:, 2, 0]) ** 2
        + (Q[:, 1, 0] - Q[:, 0, 1]) ** 2
    )
    return float(np.degrees(np.arctan2(sin_t, cos_t).min()))


# Main Problem

## Prompt

In electron backscatter diffraction (EBSD) studies of cubic polycrystals, a grain boundary is called a coincidence-site-lattice (CSL) boundary of index \\(\\Sigma\\) when the misorientation between its two grains lies within Brandon's angular tolerance of an exact \\(\\Sigma\\) rotation. Implement this classification for a boundary between two grains with known orientations. All conventions are fixed below.

**Orientations.** The orientation of a grain is given by its Bunge Euler angles \\((\\varphi_1,\\Phi,\\varphi_2)\\) in degrees. The passive orientation matrix \\(g\\) converts the sample-frame components \\(\\mathbf{v}_{\\mathrm{s}}\\) of a vector into its crystal-frame components \\(\\mathbf{v}_{\\mathrm{c}}=g\\,\\mathbf{v}_{\\mathrm{s}}\\) and is

\\[
g=\\begin{pmatrix}
\\cos\\varphi_1\\cos\\varphi_2-\\sin\\varphi_1\\sin\\varphi_2\\cos\\Phi & \\sin\\varphi_1\\cos\\varphi_2+\\cos\\varphi_1\\sin\\varphi_2\\cos\\Phi & \\sin\\varphi_2\\sin\\Phi\\\\
-\\cos\\varphi_1\\sin\\varphi_2-\\sin\\varphi_1\\cos\\varphi_2\\cos\\Phi & -\\sin\\varphi_1\\sin\\varphi_2+\\cos\\varphi_1\\cos\\varphi_2\\cos\\Phi & \\cos\\varphi_2\\sin\\Phi\\\\
\\sin\\varphi_1\\sin\\Phi & -\\cos\\varphi_1\\sin\\Phi & \\cos\\Phi
\\end{pmatrix}.
\\]

**Misorientation.** For grains A and B the misorientation matrix is \\(M=g_B\\,g_A^{\\mathsf T}\\). It maps crystal-frame components of grain A onto crystal-frame components of grain B. The crystal axes of both grains are the cube axes.

**\\(\\Sigma\\) rotations.** A proper rotation matrix \\(R\\) (\\(R^{\\mathsf T}R=I\\), \\(\\det R=+1\\)) expressed in the cube axes is called a \\(\\Sigma\\) rotation if \\(\\Sigma\\) is the smallest positive integer for which all nine entries of \\(\\Sigma R\\) are integers. Let \\(\\mathcal{S}_\\Sigma\\) be the set of all \\(\\Sigma\\) rotations. The rotation angle \\(\\theta(Q)\\in[0,\\pi]\\) of a proper rotation \\(Q\\) satisfies \\(\\cos\\theta(Q)=\\tfrac12(\\operatorname{tr}Q-1)\\). The deviation of the boundary from exact \\(\\Sigma\\) coincidence is

\\[
\\Delta\\theta(\\Sigma)=\\min_{R\\in\\mathcal{S}_\\Sigma}\\theta\\!\\left(M R^{\\mathsf T}\\right).
\\]

**Brandon's criterion.** With a constant \\(\\theta_0\\), the boundary is matched with an odd index \\(\\Sigma\\le\\Sigma_{\\max}\\) if \\(\\Delta\\theta(\\Sigma)\\le\\theta_0/\\sqrt{\\Sigma}\\) (equality counts as a match). \\(\\Sigma=1\\) represents low-angle boundaries. If several indices satisfy the criterion, the smallest one is reported.

Implement the function below. It returns the reported index and its deviation in degrees. If no odd \\(\\Sigma\\le\\Sigma_{\\max}\\) satisfies the criterion, it returns `(0, float('nan'))`. The deviation must have an absolute error below \\(10^{-8}\\) degrees. Use only NumPy and the Python standard library.

```python
import numpy as np

def classify_csl_boundary(euler_a, euler_b, sigma_max=49, theta0=15.0):
    """
    Classify a cubic grain boundary as a CSL boundary using Brandon's criterion.

    Inputs:
        euler_a : array_like of 3 floats
            Bunge Euler angles (phi1, Phi, phi2) of grain A, in degrees.
        euler_b : array_like of 3 floats
            Bunge Euler angles (phi1, Phi, phi2) of grain B, in degrees.
        sigma_max : int, default 49
            Largest CSL index considered. Only odd Sigma = 1, 3, 5, ... <= sigma_max
            are tested, in increasing order.
        theta0 : float, default 15.0
            Brandon constant in degrees. The maximum allowed deviation for index
            Sigma is theta0 / sqrt(Sigma) degrees.

    Output:
        (sigma, deviation) : tuple (int, float)
            sigma : the smallest odd Sigma <= sigma_max whose deviation satisfies
                deviation <= theta0 / sqrt(Sigma); 0 if there is no such Sigma.
            deviation : deviation of the boundary from exact Sigma coincidence, in
                degrees; float('nan') if sigma == 0.
    """
    ...
```


## Background

**CSL boundaries and Brandon's criterion.** Two cubic lattices related by a rotation \\(R\\) share a coincidence-site lattice only if \\(R\\) has rational entries in the cubic axes; the index \\(\\Sigma\\) is the smallest positive integer such that \\(\\Sigma R\\) is an integer matrix. Boundaries close to such a rotation often have low energy and special properties (for example, twins are \\(\\Sigma 3\\) boundaries), so EBSD maps are routinely screened for them. Exact coincidence never occurs in measured data, so Brandon proposed that a boundary belongs to \\(\\Sigma\\) if its angular deviation from the exact \\(\\Sigma\\) rotation does not exceed \\(15^\\circ/\\sqrt{\\Sigma}\\). Low-angle boundaries (below \\(15^\\circ\\)) then appear as \\(\\Sigma=1\\).

**Orientation matrices and misorientation.** In Bunge's convention \\(g=R_z(\\varphi_2)\\,R_x(\\Phi)\\,R_z(\\varphi_1)\\) with passive rotations, so that \\(g\\) maps sample coordinates to crystal coordinates. Equivalent crystal descriptions of a grain differ by one of the 24 proper cubic rotations \\(O\\): \\(g\\to O\\,g\\). The misorientation \\(M=g_B g_A^{\\mathsf T}\\) becomes \\(O_B M O_A^{\\mathsf T}\\) for such a change and \\(M^{\\mathsf T}\\) if the grains are swapped.

**Structure of \\(\\mathcal{S}_\\Sigma\\).** Every rational rotation matrix is generated by the Euler-Rodrigues formula from a primitive integer quaternion \\(q=(a,b,c,d)\\), \\(\\gcd(a,b,c,d)=1\\), unique up to overall sign. With \\(N=a^2+b^2+c^2+d^2\\),

\\[
R(q)=\\frac{1}{N}\\begin{pmatrix}
a^2+b^2-c^2-d^2 & 2(bc-ad) & 2(bd+ac)\\\\
2(bc+ad) & a^2-b^2+c^2-d^2 & 2(cd-ab)\\\\
2(bd-ac) & 2(cd+ab) & a^2-b^2-c^2+d^2
\\end{pmatrix},
\\]

and its index \\(\\Sigma\\) is the largest odd divisor of \\(N\\); entries of \\(N\\,R(q)\\) may share a power-of-two factor. Since each row of \\(\\Sigma R\\) is an integer vector of squared length \\(\\Sigma^2\\), even \\(\\Sigma\\) is impossible and \\(\\mathcal{S}_\\Sigma=\\varnothing\\). \\(\\mathcal{S}_1\\) is the set of the 24 proper cubic rotations. \\(\\mathcal{S}_\\Sigma\\) is closed under left and right multiplication by these 24 rotations and under transposition, and it has \\(24\\,\\psi(\\Sigma)\\) elements with \\(\\psi(\\Sigma)=\\Sigma\\prod_{p\\mid\\Sigma}(1+1/p)\\). Because of this closure, \\(\\Delta\\theta(\\Sigma)\\) is the same for all crystallographically equivalent descriptions of the boundary and for both orderings of the grains; for \\(\\Sigma=1\\) it equals the disorientation angle.

**Rotation angle.** \\(\\theta\\) is the geodesic distance on \\(SO(3)\\) between \\(M\\) and \\(R\\). Deviations far below \\(10^{-6}\\) degrees occur for numerically constructed bicrystals and have to be resolved accurately.

**Example.** The \\(\\Sigma 5\\) rotation \\(2\\arctan(1/3)\\approx36.87^\\circ\\) about \\([100]\\) has \\(\\theta_0/\\sqrt5=6.71^\\circ\\) as maximum deviation for \\(\\theta_0=15^\\circ\\).

References: D. G. Brandon, Acta Metallurgica 14, 1479-1484 (1966), https://doi.org/10.1016/0001-6160(66)90168-4; H. Grimmer, W. Bollmann and D. H. Warrington, Acta Cryst. A30, 197-207 (1974), https://doi.org/10.1107/S056773947400043X


## Testing Template

In [ ]:
import math
import numpy as np


def _rot(axis, angle_deg):
    # active rotation matrix about `axis` by `angle_deg` (Rodrigues' formula)
    n = np.asarray(axis, dtype=float)
    n = n / np.linalg.norm(n)
    t = np.radians(angle_deg)
    k = np.array([[0.0, -n[2], n[1]], [n[2], 0.0, -n[0]], [-n[1], n[0], 0.0]])
    return np.cos(t) * np.eye(3) + np.sin(t) * k + (1.0 - np.cos(t)) * np.outer(n, n)


def _bunge(euler):
    # passive Bunge orientation matrix g = Rz(phi2) Rx(Phi) Rz(phi1)
    p1, big_p, p2 = np.radians(np.asarray(euler, dtype=float))
    c1, s1, c2, s2, cc, ss = np.cos(p1), np.sin(p1), np.cos(p2), np.sin(p2), np.cos(big_p), np.sin(big_p)
    return np.array([[c1 * c2 - s1 * s2 * cc, s1 * c2 + c1 * s2 * cc, s2 * ss],
                     [-c1 * s2 - s1 * c2 * cc, -s1 * s2 + c1 * c2 * cc, c2 * ss],
                     [s1 * ss, -c1 * ss, cc]])


def _euler_from_matrix(g):
    # inverse of _bunge (for Phi = 0 only phi1 + phi2 is defined; phi2 is set to 0)
    big_p = np.arccos(np.clip(g[2, 2], -1.0, 1.0))
    if np.sin(big_p) > 1e-12:
        p1, p2 = np.arctan2(g[2, 0], -g[2, 1]), np.arctan2(g[0, 2], g[1, 2])
    else:
        p1, p2 = np.arctan2(g[0, 1], g[0, 0]), 0.0
    return np.degrees([p1, big_p, p2]) % 360.0


def _rational_rotation(m, n, uvw):
    # rotation by 2*arctan(n*|uvw|/m) about [uvw]; a Sigma rotation for integers m, n
    return _rot(uvw, 2.0 * np.degrees(np.arctan(n * np.linalg.norm(uvw) / m)))


def _partner(euler_a, misorientation):
    # Bunge angles of grain B such that g_B g_A^T equals `misorientation`
    return _euler_from_matrix(misorientation @ _bunge(euler_a))


def test_case_1():
    # exact CSL boundaries between a generic grain A and a grain B built from a known Sigma rotation
    euler_a = [37.0, 52.0, 81.0]
    cases = [((3, 1, (1, 1, 1)), 3), ((3, 1, (1, 0, 0)), 5), ((5, 1, (1, 1, 1)), 7),
             ((4, 1, (1, 1, 0)), 9), ((3, 1, (1, 1, 0)), 11), ((5, 1, (1, 0, 0)), 13),
             ((7, 1, (1, 1, 1)), 13), ((4, 1, (1, 0, 0)), 17), ((6, 1, (1, 1, 0)), 19),
             ((9, 1, (1, 1, 1)), 21), ((7, 1, (1, 0, 0)), 25), ((5, 1, (1, 1, 0)), 27),
             ((5, 2, (1, 0, 0)), 29)]
    for (m, n, uvw), expected_sigma in cases:
        euler_b = _partner(euler_a, _rational_rotation(m, n, uvw))
        sigma, deviation = classify_csl_boundary(euler_a, euler_b)
        assert sigma == expected_sigma
        assert abs(deviation) < 1e-7

    # Euler angles outside [0, 360) describe the same orientations
    euler_b = _partner(euler_a, _rational_rotation(3, 1, (1, 1, 1)))
    shifted_a = [euler_a[0] - 360.0, euler_a[1], euler_a[2] + 720.0]
    shifted_b = [euler_b[0] + 360.0, euler_b[1], euler_b[2] - 360.0]
    sigma, deviation = classify_csl_boundary(shifted_a, shifted_b)
    assert sigma == 3 and abs(deviation) < 1e-7

    # grain A with Phi = 0
    gimbal_a = [30.0, 0.0, 20.0]
    sigma, deviation = classify_csl_boundary(gimbal_a, _partner(gimbal_a, _rational_rotation(3, 1, (1, 0, 0))))
    assert sigma == 5 and abs(deviation) < 1e-7


def test_case_2():
    euler_a = [112.0, 33.0, 205.0]
    sigma5 = _rational_rotation(3, 1, (1, 0, 0))    # 36.87 deg about [100]; Brandon limit 15/sqrt(5) = 6.708 deg

    def boundary(eps):
        # Sigma 5 boundary tilted away from exact coincidence by a rotation of eps degrees
        return _partner(euler_a, _rot([2, -1, 3], eps) @ sigma5)

    # just inside and just outside the Brandon limit
    sigma, deviation = classify_csl_boundary(euler_a, boundary(6.65), sigma_max=5)
    assert sigma == 5 and abs(deviation - 6.65) < 1e-7
    sigma, deviation = classify_csl_boundary(euler_a, boundary(6.77), sigma_max=5)
    assert sigma == 0 and math.isnan(deviation)

    # near-exact coincidence: a deviation of 1e-6 deg must be resolved to better than 1e-8 deg
    for eps in [1e-6, 1e-5]:
        sigma, deviation = classify_csl_boundary(euler_a, boundary(eps), sigma_max=5)
        assert sigma == 5 and abs(deviation - eps) < 1e-8

    # the Brandon constant theta0 sets the limit
    sigma, deviation = classify_csl_boundary(euler_a, boundary(6.0), sigma_max=5, theta0=8.0)   # limit 3.58 deg
    assert sigma == 0 and math.isnan(deviation)
    sigma, deviation = classify_csl_boundary(euler_a, boundary(6.0), sigma_max=5, theta0=14.0)  # limit 6.26 deg
    assert sigma == 5 and abs(deviation - 6.0) < 1e-7

    # sigma_max limits the indices that are tested
    exact = boundary(0.0)
    for smax, expected in [(3, 0), (4, 0), (5, 5), (6, 5), (49, 5)]:
        sigma, deviation = classify_csl_boundary(euler_a, exact, sigma_max=smax)
        assert sigma == expected
        assert math.isnan(deviation) if expected == 0 else abs(deviation) < 1e-7

    # equality with the limit counts as a match: identical orientations, theta0 = 0
    sigma, deviation = classify_csl_boundary([0.0, 0.0, 0.0], [0.0, 0.0, 0.0], sigma_max=1, theta0=0.0)
    assert sigma == 1 and deviation == 0.0


def test_case_3():
    euler_a = [58.0, 71.0, 143.0]

    # exact Sigma 37 (2 arctan(1/6) about [100]) also lies within the Brandon limit of Sigma 13: smallest index is reported
    euler_b = _partner(euler_a, _rot([1, 0, 0], np.degrees(2.0 * np.arctan(1.0 / 6.0))))
    expected_dev = np.degrees(2.0 * (np.arctan(1.0 / 5.0) - np.arctan(1.0 / 6.0)))
    sigma, deviation = classify_csl_boundary(euler_a, euler_b)
    assert sigma == 13 and abs(deviation - expected_dev) < 1e-7

    # invariance: swapping the grains, and replacing either grain by a crystallographically equivalent description
    cyclic = np.array([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
    two_fold = _rot([1, 1, 0], 180.0)
    variants = [(euler_b, euler_a),
                (euler_a, _euler_from_matrix(cyclic @ _bunge(euler_b))),
                (_euler_from_matrix(two_fold @ _bunge(euler_a)), euler_b)]
    for first, second in variants:
        sigma, deviation = classify_csl_boundary(first, second)
        assert sigma == 13 and abs(deviation - expected_dev) < 1e-7

    # exact Sigma 41 (2 arctan(1/9) about [100]) is a low-angle boundary of index 1
    low = np.degrees(2.0 * np.arctan(1.0 / 9.0))
    sigma, deviation = classify_csl_boundary(euler_a, _partner(euler_a, _rot([1, 0, 0], low)))
    assert sigma == 1 and abs(deviation - low) < 1e-7

    # a 2 deg misorientation combined with a 90 deg cubic symmetry rotation is a 2 deg low-angle boundary
    sigma, deviation = classify_csl_boundary(euler_a, _partner(euler_a, _rot([0, 0, 1], 90.0) @ _rot([1, 2, 3], 2.0)))
    assert sigma == 1 and abs(deviation - 2.0) < 1e-7

    # generic pairs with default parameters (reference values verified independently)
    sigma, deviation = classify_csl_boundary([225.0, 80.7, 279.2], [81.1, 27.0, 314.5])
    assert sigma == 0 and math.isnan(deviation)
    sigma, deviation = classify_csl_boundary([328.8, 72.2, 316.0], [188.4, 82.4, 16.8])
    assert sigma == 31 and abs(deviation - 2.202497194628) < 1e-7


## Solution

In [ ]:
import numpy as np


def csl_rotation_set(sigma):
    sigma = int(sigma)
    if sigma < 1 or sigma % 2 == 0:
        return np.zeros((0, 3, 3), dtype=np.int64)

    # integer row vectors of squared length sigma^2
    r = np.arange(-sigma, sigma + 1, dtype=np.int64)
    x, y, z = np.meshgrid(r, r, r, indexing="ij")
    mask = x * x + y * y + z * z == sigma * sigma
    rows = np.stack([x[mask], y[mask], z[mask]], axis=1)

    # ordered pairs of orthogonal rows; the third row of a proper rotation is (r1 x r2) / sigma
    i_idx, j_idx = np.nonzero(rows @ rows.T == 0)
    r1, r2 = rows[i_idx], rows[j_idx]
    cross = np.cross(r1, r2)
    ok = np.all(cross % sigma == 0, axis=1)
    r1, r2, cross = r1[ok], r2[ok], cross[ok]
    mats = np.stack([r1, r2, cross // sigma], axis=1)

    # sigma must be the smallest integer that clears all denominators
    common = np.gcd.reduce(mats.reshape(len(mats), 9), axis=1)
    mats = mats[np.gcd(common, sigma) == 1]

    # ascending lexicographic order of the row-major flattened matrices
    flat = mats.reshape(len(mats), 9)
    order = np.lexsort(flat.T[::-1])
    return mats[order]


def csl_deviation(M, sigma):
    M = np.asarray(M, dtype=float)
    S = csl_rotation_set(sigma).astype(float) / sigma
    Q = np.einsum("ij,kmj->kim", M, S)  # M @ R^T for every Sigma rotation R
    cos_t = 0.5 * (Q[:, 0, 0] + Q[:, 1, 1] + Q[:, 2, 2] - 1.0)
    sin_t = 0.5 * np.sqrt(
        (Q[:, 2, 1] - Q[:, 1, 2]) ** 2
        + (Q[:, 0, 2] - Q[:, 2, 0]) ** 2
        + (Q[:, 1, 0] - Q[:, 0, 1]) ** 2
    )
    return float(np.degrees(np.arctan2(sin_t, cos_t).min()))


def classify_csl_boundary(euler_a, euler_b, sigma_max=49, theta0=15.0):
    def orientation_matrix(euler):
        p1, big_p, p2 = np.radians(np.asarray(euler, dtype=float))
        c1, s1, c2, s2 = np.cos(p1), np.sin(p1), np.cos(p2), np.sin(p2)
        cc, ss = np.cos(big_p), np.sin(big_p)
        return np.array([
            [c1 * c2 - s1 * s2 * cc, s1 * c2 + c1 * s2 * cc, s2 * ss],
            [-c1 * s2 - s1 * c2 * cc, -s1 * s2 + c1 * c2 * cc, c2 * ss],
            [s1 * ss, -c1 * ss, cc],
        ])

    misorientation = orientation_matrix(euler_b) @ orientation_matrix(euler_a).T
    for sigma in range(1, int(sigma_max) + 1, 2):
        deviation = csl_deviation(misorientation, sigma)
        if deviation <= theta0 / np.sqrt(sigma):
            return sigma, deviation
    return 0, float("nan")
